In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv("./data/train.csv")
test_df = pd.read_csv("./data/test.csv")


In [ ]:
train_df.columns

In [ ]:
test_df.columns

In [ ]:
train_df.head()

In [ ]:
train_df.set_index(train_df.PassengerId, inplace = True)

In [ ]:
train_df.head()

In [ ]:
train_df.drop('PassengerId', axis = 1, inplace = True)

In [ ]:
train_df

In [ ]:
test_df = pd.read_csv("./data/test.csv", index_col = "PassengerId")

In [ ]:
test_df.tail()

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
train_df['Survived'] = train_df['Survived'].astype("category")

In [ ]:
train_df['Survived'].dtype

In [ ]:
features = ["Pclass", "Sex", "SibSp","Parch","Embarked"]
def convert_cat(df, features):
    for feature in features:
        df[feature] = df[feature].astype("category")
convert_cat(train_df, features)
convert_cat(test_df, features)


In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
train_df.describe(include=['category'])

# EDA


In [ ]:
train_df["Survived"].value_counts().to_frame()

In [ ]:
train_df["Survived"].value_counts(normalize=True).to_frame()

In [ ]:
train_df["Sex"].value_counts(normalize=True).to_frame()

In [ ]:
sns.countplot(data= train_df, x="Sex", hue = "Survived", palette = "Blues")

In [ ]:
cols = ['Sex', 'Embarked','Pclass', 'SibSp', "Parch"]

n_rows = 2
n_cols = 3

fig, ax = plt.subplots(n_rows, n_cols, figsize = (n_cols*3.5, n_rows*3.5))
for r in range(0,n_rows):
    for c in range(0,n_cols):
        i  = r*n_cols + c
        if i < len(cols):
            ax_i = ax[r,c]
            sns.countplot(data = train_df, x = cols[i], hue = "Survived", palette = "Blues", ax = ax_i)
            ax_i.set_title(f"Figure {i+1}: Survived Rate vs {cols[i]}")
            ax_i.legend(title='', loc='upper right', labels=["Not Survived", "Survived"])
ax.flat[-1].set_visible(False)
plt.tight_layout()            

In [ ]:
sns.histplot(data=train_df, x='Age',hue="Survived", bins=40, kde=True)

In [ ]:
train_df['Fare'].describe()

In [ ]:
sns.histplot(data = train_df,hue="Survived" ,x='Fare', bins=40, palette='Blues')

In [ ]:
fare_categories = ['Economic', 'Standard', 'Expensive', 'Luxury']
quartitle = pd.qcut(train_df['Fare'], 4, labels= fare_categories)

sns.countplot(x=quartitle, hue = train_df['Survived'])

# Feature Engineering & Data Wrangling


In [ ]:
import re

def extract_title(name):
    p = re.compile(r',([\w\s]+)\.')
    match = p.search(name)
    if match:
        return match.group(1).strip()
    return None

train_df['Title'] = train_df['Name'].apply(extract_title)

In [ ]:
train_df['Title'].value_counts()

In [ ]:
test_df['Title'] = test_df['Name'].apply(extract_title)


In [ ]:
test_df['Title'].value_counts()

In [ ]:
sns.countplot(data = train_df, x='Title', hue = 'Survived')

In [ ]:
def group_title(title):
    if title in ['Mr','Miss','Mrs', 'Master']:
        return title
    elif title == 'Ms':
        return "Miss"
    else:
        return 'Others'
train_df['Title'] = train_df['Title'].apply(lambda title: group_title(title))
test_df['Title'] = test_df['Title'].apply(lambda title: group_title(title))


In [ ]:
train_df['Title'].value_counts()

In [ ]:
test_df['Title'].value_counts()

In [ ]:
sns.countplot(data = train_df, x='Title', hue = 'Survived')

In [ ]:
train_df['Family_size'] = train_df['SibSp'].astype('int') + train_df['Parch'].astype('int') + 1

In [ ]:
test_df['Family_size'] = test_df['SibSp'].astype('int') + test_df['Parch'].astype('int') + 1

In [ ]:
train_df['Family_cat'] = pd.cut(train_df['Family_size'], bins = [0,1,4,6,20], labels = ['Alone', 'Small', 'Medium', 'Large']) #(0,1], (1,4],(4,6], (6,20])
test_df['Family_cat'] = pd.cut(test_df['Family_size'], bins = [0,1,4,6,20], labels = ['Alone', 'Small', 'Medium', 'Large']) #(0,1], (1,4],(4,6], (6,20])

In [ ]:
sns.countplot(data = train_df, x= 'Family_cat', hue = 'Survived')

 ## Data wrangling
 

In [ ]:
num_features = ['Age', 'Fare']
cat_features = ['Sex', 'Pclass', 'Embarked', 'Title', 'Family_cat']
feature_cols = num_features + cat_features
print(feature_cols)

In [ ]:
def count_missing(df, feature_cols):
    n_rows = df.shape[0]
    for col in feature_cols:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            print(f"{col} has {missing_count*100/n_rows:.2f}% missing values.")
count_missing(train_df,feature_cols)
count_missing(test_df,feature_cols)

In [ ]:
age_by_sex_pclass = train_df.groupby(['Sex', 'Pclass'])['Age'].median().reset_index()
age_by_sex_pclass

In [ ]:
train_df['Age'] = train_df.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
 

In [ ]:
count_missing(train_df,feature_cols)


In [ ]:
test_df['Age'] = test_df.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
count_missing(test_df,feature_cols)


In [ ]:
x = train_df[feature_cols]
y = train_df['Survived']


In [ ]:
x_test = test_df[feature_cols]

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_transformers = Pipeline(steps =[
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])

cat_transformers = Pipeline(steps =[
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('scaler', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
preprocessor = ColumnTransformer(transformers= [
    ('num', num_transformers, num_features),
    ('cat', cat_transformers, cat_features)
])

In [ ]:
preprocessor.fit(x)

In [ ]:
X = preprocessor.transform(x)


In [ ]:
X_test = preprocessor.transform(x_test)

# Model training



In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2)

In [ ]:
X_train.shape, X_val.shape

In [ ]:
X_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


In [ ]:
log_reg = LogisticRegression(solver='liblinear', max_iter=1000)
log_reg.fit(X_train,y_train)

In [ ]:
log_reg.score(X_val,y_val)

In [ ]:
y_pred = log_reg.predict(X_val)

In [ ]:
print(classification_report(y_val, y_pred))